# Assignment 4 - Problem 1: TF-IDF Vectorization with NLTK Preprocessing

This notebook implements TF-IDF vectorization with NLTK preprocessing, demonstrating vocabulary transfer and OOV handling.

**Requirements**: Place `large.md` and `small.md` in the same directory as this notebook.

**Note**: Dependencies (numpy, scikit-learn, nltk) are auto-installed in the next cell.

In [1]:
# Install required dependencies
import sys
import subprocess

# Map package names to import names
packages = {
    'numpy': 'numpy',
    'scikit-learn': 'sklearn',
    'nltk': 'nltk'
}

print("Checking dependencies...")
missing = []
for pip_name, import_name in packages.items():
    try:
        __import__(import_name)
        print(f"✓ {pip_name}")
    except ImportError:
        print(f"✗ {pip_name} (needs installation)")
        missing.append(pip_name)

if missing:
    print(f"\nInstalling {len(missing)} package(s)...")
    for package in missing:
        # Try multiple installation methods
        success = False
        
        # Method 1: python -m pip
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package], 
                                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print(f"✓ Installed {package}")
            success = True
        except:
            pass
        
        # Method 2: pip command directly
        if not success:
            try:
                subprocess.check_call(["pip", "install", package],
                                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                print(f"✓ Installed {package}")
                success = True
            except:
                pass
        
        # Method 3: pip3 command
        if not success:
            try:
                subprocess.check_call(["pip3", "install", package],
                                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                print(f"✓ Installed {package}")
                success = True
            except:
                pass
        
        if not success:
            print(f"✗ Failed to install {package}")
            print(f"  Please run manually: pip install {package}")
            raise ImportError(f"Could not install {package}. Please install manually and restart kernel.")

print("\nAll dependencies ready!")

Checking dependencies...
✓ numpy
✓ scikit-learn
✓ nltk

All dependencies ready!


In [2]:
# Imports and Setup
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import warnings
import ssl

warnings.filterwarnings('ignore')

# Compile regex pattern once for efficiency
ALPHA_PATTERN = re.compile(r'[^a-zA-Z\s]+')

# Fix SSL certificate issue (common on macOS)
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# Download required NLTK data
print("Downloading NLTK resources...")
for resource in ['punkt', 'wordnet', 'omw-1.4']:
    try:
        nltk.data.find(resource)
        print(f"✓ {resource} already exists")
    except LookupError:
        print(f"Downloading {resource}...")
        try:
            nltk.download(resource, quiet=True)
            print(f"✓ {resource} downloaded")
        except Exception as e:
            print(f"✗ Error downloading {resource}: {e}")

def preprocess_text(text, lemmatizer):
    """
    Apply tokenization and lemmatization to text.
    
    Args:
        text (str): Input text to preprocess
        lemmatizer: NLTK WordNetLemmatizer instance
    
    Returns:
        str: Preprocessed text with tokens lemmatized and joined
    """
    # Convert to lowercase and remove non-alphabetic characters
    text = text.lower()
    text = ALPHA_PATTERN.sub(' ', text)  # Use pre-compiled pattern
    
    # Tokenize and lemmatize
    tokens = word_tokenize(text)
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
    return ' '.join(lemmatized_tokens)

print("✓ Setup complete!")

✓ punkt downloaded
✓ wordnet downloaded
✓ omw-1.4 downloaded
✓ Setup complete!


In [3]:
# Main Execution
print("="*80)
print("Assignment 4 - Problem 1: TF-IDF Vectorization with NLTK Preprocessing")
print("="*80)

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Part (a): Apply word tokenization and lemmatization
print("\n" + "="*80)
print("PART (a): Word Tokenization and Lemmatization")
print("="*80)

# Read the large text dataset with error handling
try:
    with open('large.md', 'r', encoding='utf-8') as f:
        large_text = f.read()
except FileNotFoundError:
    print("Error: large.md file not found. Please ensure the file exists.")
except Exception as e:
    print(f"Error reading large.md: {e}")

# Read the smaller text with new words
try:
    with open('small.md', 'r', encoding='utf-8') as f:
        small_text = f.read()
except FileNotFoundError:
    print("Error: small.md file not found. Please ensure the file exists.")
except Exception as e:
    print(f"Error reading small.md: {e}")

print(f"\nText lengths: Large={len(large_text)} chars, Small={len(small_text)} chars")

# Preprocess both texts
processed_large_text = preprocess_text(large_text, lemmatizer)
processed_small_text = preprocess_text(small_text, lemmatizer)

# Display samples of the processed SMALLER text
small_words = processed_small_text.split()
print("\n--- Processed Smaller Text Samples ---")
print(f"Total tokens: {len(small_words)}")
print(f"First 30 tokens: {' '.join(small_words[:30])}")
print(f"Last 30 tokens: {' '.join(small_words[-30:])}")

# Vocabulary overlap analysis
large_vocab = set(processed_large_text.split())
small_vocab = set(small_words)
new_words_in_small = small_vocab - large_vocab
print(f"\nVocabulary: {len(small_vocab)} unique tokens")
print(f"New words not in large text: {len(new_words_in_small)}")

# Part (b): Apply TF-IDF vectorization using Scikit-learn
print("\n" + "="*80)
print("PART (b): TF-IDF Vectorization")
print("="*80)

# Create TF-IDF vectorizer
# Note: Since we need multiple documents for IDF calculation,
# we split the large text into chunks
# Using fixed-size chunks for consistency with original implementation
words = processed_large_text.split()
chunk_size = 100  # Fixed chunk size for reproducibility
large_chunks = [' '.join(words[i:i+chunk_size])
               for i in range(0, len(words), chunk_size)]

print(f"\nCreated {len(large_chunks)} document chunks for IDF calculation")

tfidf_vectorizer = TfidfVectorizer(
    preprocessor=lambda x: x,  # Already preprocessed
    tokenizer=lambda x: x.split(),  # Simple split
    max_features=500,
    min_df=1,
    max_df=0.9,
)

# Fit the TF-IDF vectorizer on the large text dataset
tfidf_vectorizer.fit(large_chunks)
print(f"\nVocabulary size: {len(tfidf_vectorizer.vocabulary_)}")

# Apply the trained TF-IDF vectorizer to the smaller text
small_text_tfidf = tfidf_vectorizer.transform([processed_small_text])

# Display TF-IDF representation
print(f"\nTF-IDF representation shape: {small_text_tfidf.shape}")
print(f"Number of non-zero features: {small_text_tfidf.nnz}")

# Get top TF-IDF features for the smaller text
feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_scores = small_text_tfidf.toarray()[0]
top_indices = np.argsort(tfidf_scores)[::-1][:10]

print("\n--- Top 10 TF-IDF features in smaller text ---")
for i, idx in enumerate(top_indices, 1):
    if tfidf_scores[idx] > 0:
        print(f"{i}. '{feature_names[idx]}': {tfidf_scores[idx]:.3f}")

# Verify L2 normalization
l2_norm = np.linalg.norm(tfidf_scores)
print(f"\nL2 norm of TF-IDF vector: {l2_norm:.6f} (should be ~1.0)")


# Part (c): Information transfer analysis
print("\n" + "="*80)
print("PART (c): Information Transfer from Large to Small Text")
print("="*80)
print("""
Information transferred from large text to small text TF-IDF representation:

1. **Vocabulary**: Only terms from the large text can have non-zero TF-IDF values.
2. **IDF weights**: Document frequency statistics computed from large text corpus.
3. **Feature space**: The 500 selected features based on large text term frequencies.
4. **Normalization**: L2 normalization scheme from the training corpus.
""")

# Part (d): Handling of new words
print("\n" + "="*80)
print("PART (d): How Scikit-learn's TF-IDF Handles New Words")
print("="*80)
print("""
How Scikit-learn's TF-IDF handles new words:

• New words are silently ignored (no error or warning)
• They receive TF-IDF value of 0 (no contribution to vector)
• No <OOV> token is used to represent unknown words
• This causes information loss for out-of-vocabulary terms
""")

Assignment 4 - Problem 1: TF-IDF Vectorization with NLTK Preprocessing

PART (a): Word Tokenization and Lemmatization

Text lengths: Large=16480 chars, Small=2102 chars

--- Processed Smaller Text Samples ---
Total tokens: 300
First 30 tokens: e overall training recipe large scale pretraining similar to previous vla method we construct a large scale dataset of over k trajectory by combining diverse open source datasets such a
Last 30 tokens: with the ability to integrate perception understanding and action generation from multisensory input in the real physical world during inference we employ ddim with n sampling step e g n

Vocabulary: 172 unique tokens
New words not in large text: 49

PART (b): TF-IDF Vectorization

Created 23 document chunks for IDF calculation

Vocabulary size: 500

TF-IDF representation shape: (1, 500)
Number of non-zero features: 109

--- Top 10 TF-IDF features in smaller text ---
1. 'training': 0.371
2. 'in': 0.241
3. 'loss': 0.217
4. 'we': 0.205
5. 'a': 0.197
